# The Four Men, Hilaire Belloc

## Pre-Execution

### Imports

In [1]:
import pathlib
import pathlib as p
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import get_ident

import processfiles

### Logging

In [2]:
# %load load.py
class ThreadFilter:
    def __init__(self, id):
        self.id = id

    def filter(self, record):
        return record.thread == self.id

import logging
import sys

sysout_handler = logging.StreamHandler(sys.stdout)
sysout_handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s-%(name)s-%(levelname)s-%(filename)s-%(lineno)s-%(funcName)s-%(message)s')
sysout_handler.setFormatter(formatter)

processfiles.logger.setLevel(logging.DEBUG)

for h in processfiles.logger.handlers:
    processfiles.logger.removeHandler(h)
processfiles.logger.addHandler(sysout_handler)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
for h in logger.handlers:
    logger.removeHandler(h)
logger.addHandler(sysout_handler)

## Variables

In [3]:
bookpath = "/home/linuxuser/PGDP/belloc-the-four-men/"
sourcefiles = pathlib.Path(bookpath, "fourmenfarrago00belluoft_jp2")

## Variables

#### Globals

In [4]:
# B/W Threshold Level
text_threshold = 140

default_fuzzy_pct = 0.02
default_pixel_count_columns = 100
default_pixel_count_rows = 50

# "Regular" text on the page ratio
# Default = 1.65
# page_h_w_ratio = 1.65

page_h_w_ratio = 1.65

proof_start_idx0 = 4
proof_end_idx0 = 389

cover_idx0 = 0
title_idx0 = 8

frontmatter_start_idx0 = proof_start_idx0
frontmatter_end_idx0 = 15

bodymatter_start_idx0 = frontmatter_end_idx0 + 1
bodymatter_end_idx0 = proof_end_idx0

frontmatter_page_nbr_start = 1
bodymatter_page_nbr_start = 11

#### Plates

In [5]:
plate_pages_b = [6]
plate_pages_p = [7]
plate_pages_r = []

#### Blank Pages

In [6]:
non_plate_blank_pages = [
    11, 17, 27, 29, 95, 171, 173, 361, 381
]
blank_pages = [
    *non_plate_blank_pages,
    *plate_pages_b,
    *plate_pages_r,
]

#### Text Alignment Per Page

In [7]:
align_top_pages = [
    4,14,26,170,359,378
]
align_center_pages = [
    *plate_pages_p,
    5,8,9,10,15,16,28,93,94,172,360,379,380
]
align_bottom_pages = [
    12,18,30,96,174,362,382
]

In [8]:

rotated_standard_pages = [
]

single_dimension_rescale = [
]

# Additional Margins - add additional margin to crop prior to resizing
# % is based on amount of text in crop
# Dict Tuples "Page" : (left %, right %, top %, bottom %) percents as decimals
# Use for Pages where the text isn't justified (left/right margin not the same - poetry etc)
# Also for anywhere special structure is needed
white_space_additional = {
    4: (0.9, 0, 0, 0),
    9: (0.2, 0.2, 0, 0),
    10: (0.1, 0.1, 0, 0),
    15: (0.3, 0.3, 0, 0),
    379: (0.3, 0.3, 0, 0),
    380: (1.1, 1.1, 0, 0),
}

# Fuzzy edge finding adjustment (pct, pixel count columns, pixel count rows)
# Adjust to (0.01, 1, 1) to have first pixel count as the edge
edge_finding_adjust = {
#    9: (0.01, 1, 1),
}

threshold_level_adjust = {}

do_morph = []
skip_denoise = []

# Split sections (e.g. indexes)
split_page_sections = {
    # "p239": [
    #     {"suffix": "ch", "T": 0, "B": 610, "crop_in_standard_page": True},
    #     {"suffix": "cl", "T": 610, "B": 2270, "R": 740},
    #     {"suffix": "cr", "T": 610, "B": 2270, "L": 740},
    # ],
    # "p256": [
    #     {"suffix": "acl", "T": 140, "B": 870, "R": 830},
    #     {"suffix": "acr", "T": 140, "B": 870, "L": 830},
    #     {"suffix": "bch", "T": 870, "B": 1035, "crop_in_standard_page": True},
    #     {"suffix": "bcl", "T": 1035, "B": 2230, "R": 830},
    #     {"suffix": "bcr", "T": 1035, "B": 2220, "L": 830},
    #     {"suffix": "bcz", "T": 2228, "crop_in_standard_page": True},
    # ],
}

ocr_crop_skip = [
    *blank_pages,
    *plate_pages_p,
    *align_center_pages,
    *align_bottom_pages,
    *rotated_standard_pages,
    *single_dimension_rescale,
]
ocr_crop_top = 0
ocr_crop_bottom = 0
ocr_crop_left = 0
ocr_crop_right = 0

deskew_before_crop = {}
deskew_after_crop = {}

skip_auto_deskew = [
    *single_dimension_rescale,
    *align_top_pages,
    *align_center_pages,
    *align_bottom_pages,
    33,
    114,
    124,
    166,
    199,
    264,
    322,
    355,
    356,
    377,
    386,
]

# intital page crop {idx0 or "page": (L, R, T, B)}
initial_crop_all = (15, 15, 100, 100)

# "odd" page numbers
right_pages = [
    l
    for l in list(range(0, proof_end_idx0+1, 2))
    if l >= proof_start_idx0 and l <= proof_end_idx0
]

# "even" page numbers
left_pages = [
    l
    for l in list(range(1, proof_end_idx0+1, 2))
    if l >= proof_start_idx0 and l <= proof_end_idx0
]

right_initial_crop = {}
left_initial_crop = {}
#right_initial_crop = {l: (250, 50, 20, 250) for l in right_pages}
#left_initial_crop = {l: (50, 250, 20, 250) for l in left_pages}

initial_crop = {
    **right_initial_crop,
    **left_initial_crop,
    33: (0,0,0,0),
    199: (0,0,0,0),
    262: (0,0,0,0),
    264: (0,0,0,0),
    382: (0,0,0,0),
    383: (0,0,0,0),
    384: (0,0,0,0),
    385: (0,0,0,0),
    386: (0,0,0,0),
    387: (0,0,0,0),
    388: (0,0,0,0),
    389: (0,0,0,0),
}

#initial_crop[35] = (50, 0, 150, 100)
#initial_crop[246] = (0,0,0,0)

#initial_crop[61] = (50, 250, 20, 250)

# for p in ["p063", "p084", "p144", "p176", "p268", "p270", "p260", "p239"]:
#     initial_crop[p] = (
#         initial_crop[p][0],
#         initial_crop[p][1],
#         initial_crop[p][2],
#         initial_crop[p][3] - 50,
#     )

#initial_crop = {
#}

### Basic Working Paths

In [9]:
workpath = pathlib.Path(bookpath, "processing")
pathlib.Path.mkdir(self=pathlib.Path(workpath), exist_ok=True)

logpath = pathlib.Path(workpath, "logs")
pathlib.Path.mkdir(self=pathlib.Path(logpath), exist_ok=True)

logfile = pathlib.Path(logpath, "0_full_log.log")
log_file_handler = logging.FileHandler(filename=logfile, mode="w", encoding="utf-8")
log_file_handler.setFormatter(formatter)
log_file_handler.setLevel(logging.DEBUG)

processfiles.logger.addHandler(log_file_handler)
logger.addHandler(log_file_handler)

jp2path = pathlib.Path(workpath, "processed_jp2")
pathlib.Path.mkdir(self=pathlib.Path(jp2path), exist_ok=True)

jpgpath = pathlib.Path(workpath, "processed_jpg")
pathlib.Path.mkdir(self=pathlib.Path(jpgpath), exist_ok=True)

hi_res_jpg_path = pathlib.Path(workpath, "hi_res_jpg")
pathlib.Path.mkdir(self=pathlib.Path(hi_res_jpg_path), exist_ok=True)

original_as_png_path = pathlib.Path(workpath, "original_as_png")
pathlib.Path.mkdir(self=pathlib.Path(original_as_png_path), exist_ok=True)

original_override_png_path = pathlib.Path(workpath, "original_override_png")
pathlib.Path.mkdir(self=pathlib.Path(original_override_png_path), exist_ok=True)

original_override_jpg_path = pathlib.Path(workpath, "original_override_jpg")
pathlib.Path.mkdir(self=pathlib.Path(original_override_jpg_path), exist_ok=True)

original_thumbnail_path = pathlib.Path(workpath, "original_thumbnail")
pathlib.Path.mkdir(self=original_thumbnail_path, exist_ok=True)

gegl_c2g_grayscale_png_path = pathlib.Path(workpath, "gegl_c2g_grayscale_png")
pathlib.Path.mkdir(self=gegl_c2g_grayscale_png_path, exist_ok=True)

grayscale_override_png_path = pathlib.Path(workpath, "grayscale_override_png")
pathlib.Path.mkdir(self=pathlib.Path(grayscale_override_png_path), exist_ok=True)

grayscale_override_jpg_path = pathlib.Path(workpath, "grayscale_override_jpg")
pathlib.Path.mkdir(self=pathlib.Path(grayscale_override_jpg_path), exist_ok=True)

index_processed_png_path = pathlib.Path(workpath, "index_processed_png")
pathlib.Path.mkdir(self=pathlib.Path(index_processed_png_path), exist_ok=True)

split_processed_png_path = pathlib.Path(workpath, "split_processed_png_path")
pathlib.Path.mkdir(self=pathlib.Path(split_processed_png_path), exist_ok=True)

proofing_images_png_path = pathlib.Path(workpath, "proofing_images_png")
pathlib.Path.mkdir(self=proofing_images_png_path, exist_ok=True)

pre_ocr_images_png_path = pathlib.Path(workpath, "pre_ocr_images_png")
pathlib.Path.mkdir(self=pre_ocr_images_png_path, exist_ok=True)

ocr_images_png_path = pathlib.Path(workpath, "ocr_images_png")
pathlib.Path.mkdir(self=ocr_images_png_path, exist_ok=True)

ocr_texts_path = pathlib.Path(workpath, "ocr_text")
pathlib.Path.mkdir(self=ocr_texts_path, exist_ok=True)

debug_png_path = pathlib.Path(workpath, "debug_png")
pathlib.Path.mkdir(self=debug_png_path, exist_ok=True)

for_zip_path = pathlib.Path(workpath, "for_zip")
pathlib.Path.mkdir(self=for_zip_path, exist_ok=True)


### Create Initial Variables for Processing the book

In [10]:
source_files = list(sorted([f for f in sourcefiles.iterdir() if f.is_file()]))

source_file_count = len(source_files)
prefix_format_string = "{:0>3d}" if source_file_count < 1000 else "{:0>4d}"


original_override_jpg_files = [
    v for v in list(
        sorted([f for f in original_override_jpg_path.iterdir() if f.is_file()])
    )
]
original_override_png_files = [
    v for v in list(
        sorted([f for f in original_override_png_path.iterdir() if f.is_file()])
    )
]
grayscale_override_jpg_files = [
    v for v in list(
        sorted([f for f in grayscale_override_jpg_path.iterdir() if f.is_file()])
    )
]
grayscale_override_png_files = [
    v for v in list(
        sorted([f for f in grayscale_override_png_path.iterdir() if f.is_file()])
    )
]
overrides = [
    original_override_jpg_files,
    original_override_png_files,
    grayscale_override_jpg_files,
    grayscale_override_png_files,
]

processing_dict = {
    idx0: {"name": n.name, "stem": n.stem, "original_file": n, "outputs": []}
    for idx0, n in list(enumerate(source_files))
}

fidx = frontmatter_page_nbr_start
pidx = bodymatter_page_nbr_start


unnumbered_plate_pages = list(sorted([*plate_pages_b, *plate_pages_p, *plate_pages_r]))

for k in list(processing_dict.keys()):
    prefix = ""
    ignore = False
    if k < proof_start_idx0:
        prefix = f"aNotForProofing{k}"
        ignore = True
    elif k > proof_end_idx0:
        prefix = f"xNotForProofing{k}"
        ignore = True

    if k >= frontmatter_start_idx0 and k <= frontmatter_end_idx0:
        prefix = ("f" + prefix_format_string).format(fidx)
        if k not in unnumbered_plate_pages:
            fidx = fidx + 1
    if k >= bodymatter_start_idx0 and k <= bodymatter_end_idx0:
        prefix = ("p" + prefix_format_string).format(pidx)
        if k not in unnumbered_plate_pages:
            pidx = pidx + 1

    if k in unnumbered_plate_pages:
        if k in plate_pages_b:
            prefix = prefix + "b"
        if k in plate_pages_p:
            prefix = prefix + "p"
        if k in plate_pages_r:
            prefix = prefix + "r"

    processing_dict[k]["prefix"] = prefix
    processing_dict[k]["ignore"] = ignore

    if not ignore:
        processing_dict[k]["blank"] = (
            True
            if k in blank_pages or processing_dict[k]["prefix"] in blank_pages
            else False
        )
        processing_dict[k]["skip_ocr_crop"] = (
            True
            if (k in ocr_crop_skip or processing_dict[k]["prefix"] in ocr_crop_skip)
            else False
        )

        for o in original_override_jpg_files:
            if processing_dict[k]["stem"] in o.stem:
                processing_dict[k]["original_override_jpg_file"] = o
        for o in original_override_png_files:
            if processing_dict[k]["stem"] in o.stem:
                processing_dict[k]["original_override_png_file"] = o
        for o in grayscale_override_jpg_files:
            if processing_dict[k]["stem"] in o.stem:
                processing_dict[k]["grayscale_override_jpg_file"] = o
        for o in grayscale_override_png_files:
            if processing_dict[k]["stem"] in o.stem:
                processing_dict[k]["grayscale_override_png_file"] = o

        keys = [k, prefix]
        if not any(k in keys for k in list(split_page_sections.keys())):
            processing_dict[k]["outputs"].append(
                {
                    "full_prefix": prefix,
                    "full_page": True,
                },
            )
            processing_dict[k]["outputs"][0]["pre_ocr_target_file"] = pathlib.PosixPath(
                pre_ocr_images_png_path,
                processing_dict[k]["stem"] + "_" + prefix + ".png",
            )
            processing_dict[k]["outputs"][0]["ocr_target_file"] = pathlib.PosixPath(
                ocr_images_png_path, processing_dict[k]["stem"] + "_" + prefix + ".png"
            )
            processing_dict[k]["outputs"][0]["ocr_target_text_file"] = (
                pathlib.PosixPath(
                    ocr_texts_path, processing_dict[k]["stem"] + "_" + prefix + ".txt"
                )
            )
            processing_dict[k]["outputs"][0]["proofing_image"] = pathlib.PosixPath(
                proofing_images_png_path,
                processing_dict[k]["stem"] + "_" + prefix + ".png",
            )

        else:
            split_list = split_page_sections.get(k, split_page_sections["prefix"])
            split_idx = 0
            for s in split_list:
                full_prefix = prefix + s["suffix"]
                processing_dict[k]["outputs"].append(
                    {
                        "full_prefix": full_prefix,
                        "full_page": False,
                    },
                )
                processing_dict[k]["outputs"][split_idx]["pre_ocr_target_file"] = (
                    pathlib.PosixPath(
                        pre_ocr_images_png_path,
                        processing_dict[k]["stem"] + "_" + full_prefix + ".png",
                    )
                )
                processing_dict[k]["outputs"][split_idx]["ocr_target_file"] = (
                    pathlib.PosixPath(
                        ocr_images_png_path,
                        processing_dict[k]["stem"] + "_" + full_prefix + ".png",
                    )
                )
                processing_dict[k]["outputs"][split_idx]["ocr_target_text_file"] = (
                    pathlib.PosixPath(
                        ocr_texts_path,
                        processing_dict[k]["stem"] + "_" + full_prefix + ".txt",
                    )
                )
                processing_dict[k]["outputs"][split_idx]["proofing_image"] = (
                    pathlib.PosixPath(
                        proofing_images_png_path,
                        processing_dict[k]["stem"] + "_" + full_prefix + ".png",
                    )
                )
                split_idx = split_idx + 1


def has_gap(A):
    "Return maximum j-i subject to A[i] <= A[j]."
    prev_n = A[0]
    for n in A[1:]:
        if prev_n + 1 != n:
            logger.error(f"Missing Page {n}")
            return True
        prev_n = n
    return False


if has_gap(list(sorted(processing_dict.keys()))):
    logger.error("Missing page!")

In [11]:
processing_dict

{0: {'name': 'fourmenfarrago00belluoft_0001.jp2',
  'stem': 'fourmenfarrago00belluoft_0001',
  'original_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0001.jp2'),
  'outputs': [],
  'prefix': 'aNotForProofing0',
  'ignore': True},
 1: {'name': 'fourmenfarrago00belluoft_0002.jp2',
  'stem': 'fourmenfarrago00belluoft_0002',
  'original_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0002.jp2'),
  'outputs': [],
  'prefix': 'aNotForProofing1',
  'ignore': True},
 2: {'name': 'fourmenfarrago00belluoft_0003.jp2',
  'stem': 'fourmenfarrago00belluoft_0003',
  'original_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0003.jp2'),
  'outputs': [],
  'prefix': 'aNotForProofing2',
  'ignore': True},
 3: {'name': 'fourmenfarrago00belluoft_0004.jp2',
  'stem': 'fourmenfarrago00belluoft_0004',
  'original_f

### Step 0: Skip Steps during in-progress work

In [12]:
skip_steps = [1,2,3,4,5,6,7,8,9,10,11]
logger.info(f"Skipping Steps {skip_steps}")

2024-12-20 07:09:47,648-root-INFO-2733138965.py-2-<module>-Skipping Steps [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


### Step 1a: Convert JP2 to JPEG if source file is JP2


In [13]:
step_id = 1
if step_id in skip_steps:
    logger.info("Skipping Step 1")
else:

    def convert_source_jp2_file(idx):
        print(f"converting {idx}")
        source_file_path = processing_dict[idx]["original_file"]
        target_file_path = pathlib.Path(
            original_override_jpg_path,
            processing_dict[idx]["stem"]
            + "_"
            + processing_dict[idx]["prefix"]
            + ".jpg",
        )
        text = f"idx: {idx} Source: {source_file_path} Target: {target_file_path}"
        logger.info(f"Processing: {text}")
        processfiles.convert_jp2_file_to_jpg_file(
            source_file_path=source_file_path, target_file_path=target_file_path
        )

        processing_dict[k]["original_override_jpg_file"] = target_file_path
        return f"Completed: {text}"

    with ThreadPoolExecutor(max_workers=8) as executor:
        thumbnails = [
            executor.submit(convert_source_jp2_file, idx)
            for idx in processing_dict.keys()
            if processing_dict[idx]["original_file"].suffix == ".jp2"
        ]
        for future in as_completed(thumbnails):
            logger.info(future.result())

2024-12-20 07:09:47,662-root-INFO-716182006.py-3-<module>-Skipping Step 1


In [14]:
processing_dict[25]

{'name': 'fourmenfarrago00belluoft_0026.jp2',
 'stem': 'fourmenfarrago00belluoft_0026',
 'original_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0026.jp2'),
 'outputs': [{'full_prefix': 'p020',
   'full_page': True,
   'pre_ocr_target_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/pre_ocr_images_png/fourmenfarrago00belluoft_0026_p020.png'),
   'ocr_target_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_images_png/fourmenfarrago00belluoft_0026_p020.png'),
   'ocr_target_text_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0026_p020.txt'),
   'proofing_image': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/proofing_images_png/fourmenfarrago00belluoft_0026_p020.png')}],
 'prefix': 'p020',
 'ignore': False,
 'blank': False,
 'skip_ocr_crop': False,
 'original_override_jpg_file': PosixPath('/home/linuxuser

### Step 2: Create Large GrayScale JPG Files

This is a VERY long operation, ~30 seconds *per* image

Skip Blank Pages

In [15]:
step_id = 2

create_grayscale = False

def process_to_grayscale_with_gegl(k, source_image_file):
    target_image_file = pathlib.PosixPath(
        grayscale_override_jpg_path, processing_dict[k]["stem"] + ".jpg"
    )
    if create_grayscale:
        processing_dict[k]["grayscale_override_jpg_file"] = target_image_file

    if step_id not in skip_steps:
        processfiles.run_gegl_c2g(
            source_image_file=source_image_file,
            target_image_file=target_image_file,
        )
        return f"Completed: {target_image_file}"

with ThreadPoolExecutor(max_workers=1) as executor:
    grayscale_images = [
        executor.submit(process_to_grayscale_with_gegl, idx, processing_dict[idx]["original_override_jpg_file"])
        for idx, f in sorted(enumerate(processing_dict))
        if idx not in blank_pages and f not in blank_pages
        and not processing_dict[f]["ignore"]
    ]
    for future in as_completed(grayscale_images):
        logger.info(future.result())

2024-12-20 07:09:47,695-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,703-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,704-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,705-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,706-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,706-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,707-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,708-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,708-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,710-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,711-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,712-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,712-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,713-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,713-root-INFO-1835355026.py-27-<module>-None
2024-12-20 07:09:47,714-r

### Step 3: Split Pages TODO UPDATE

In [16]:
if True: # or 3 in skip_steps or len(split_page_sections.keys()) == 0 or not split_page_sections:
    logger.info("Skipping Step 3")
else:
    for k in list(split_page_sections.keys()):
        # TODO

        png_file_path = pathlib.Path(original_as_png_path, f"{prefix}.png")
        gegl_c2g_file_path = pathlib.Path(
            gegl_c2g_grayscale_png_path, f"{prefix}.png"
        )
        grayscale_override_file_path = pathlib.Path(
            grayscale_override_png_path, f"{prefix}.png"
        )
        if gegl_c2g_file_path.exists():
            png_file_path = gegl_c2g_file_path
        if grayscale_override_file_path.exists():
            png_file_path = grayscale_override_file_path

        logging.debug(f"Processing {prefix}")
        img = processfiles.load_image(src=png_file_path)

        for p in split_page_sections[prefix]:
            full_prefix = "{}{}".format(prefix, p["suffix"])
            logging.debug(f"Creating {full_prefix}")
            new_img = img.copy()
            h, w = new_img.shape[:2]
            T = p.get("T", 0)
            B = p.get("B", h)
            L = p.get("L", 0)
            R = p.get("R", w)
            new_img = new_img[T:B, L:R]
            processfiles.write_png(
                img=new_img,
                f=pathlib.Path(split_processed_png_path, f"{full_prefix}.png"),
            )


2024-12-20 07:09:47,966-root-INFO-3193673485.py-2-<module>-Skipping Step 3


### Step 4: Thumbnails

Create small JPGs from the big PNGs

In [17]:
step_id = 4

def create_thumbnail(idx):
    processing_dict[idx]["original_thumbnail"] = pathlib.Path(
        original_thumbnail_path,
        processing_dict[idx]["original_file"].stem + ".jpg",
    )

    if step_id not in skip_steps:
        logger.info(f'Start Thumbnail For: {processing_dict[idx]["original_file"]}')
        processfiles.create_file_thumbnail(
            source_file_path=processing_dict[idx]["original_file"],
            target_file_path=processing_dict[idx]["original_thumbnail"],
            jpeg_quality=100,
            max_dimension=800,
        )
        return f'Created Thumbnail: {processing_dict[idx]["original_thumbnail"]}'
    return f'Skipped Creating Thumbnail For: {processing_dict[idx]["original_file"]}'


with ThreadPoolExecutor() as executor:
    thumbnails = [
        executor.submit(create_thumbnail, idx)
        for idx in processing_dict.keys()
        # if idx in (10,)
    ]
    for future in as_completed(thumbnails):
        logger.info(future.result())

2024-12-20 07:09:48,002-root-INFO-1834518277.py-28-<module>-Skipped Creating Thumbnail For: /home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0061.jp2
2024-12-20 07:09:48,005-root-INFO-1834518277.py-28-<module>-Skipped Creating Thumbnail For: /home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0294.jp2
2024-12-20 07:09:48,006-root-INFO-1834518277.py-28-<module>-Skipped Creating Thumbnail For: /home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0030.jp2
2024-12-20 07:09:48,006-root-INFO-1834518277.py-28-<module>-Skipped Creating Thumbnail For: /home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0138.jp2
2024-12-20 07:09:48,007-root-INFO-1834518277.py-28-<module>-Skipped Creating Thumbnail For: /home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0232.jp2
2024-12-20 07:09:48,

### 5: Create Proofing Images

Run over all source and/or the override images

#### 5a. Define process_proofing_image function   

In [18]:
def process_proofing_image_from_source(
    prefix,
    idx,
    proofing_source_file_path: pathlib.Path,
    proofing_image: pathlib.Path,
    pre_ocr_target_file: pathlib.Path,
    debug=False,
    optimize_png=True,
):
    text = f"idx: {idx} Prefix: {prefix} Source File: {proofing_source_file_path}"

    logger.info(f"Processing: {text}")

    def dict_multi_get(d: dict, keys: list, default=None):
        result = default
        for k in keys:
            if get_value := d.get(k):
                result=get_value
                break
        return result

    keys = [prefix, idx]

    # Additional Whitespace
    white_space_additional_value = dict_multi_get(white_space_additional, keys)
    logger.info(f"Additional Whitespace: {white_space_additional_value}")

    # Adjust Threshold Value
    threshold_level = dict_multi_get(threshold_level_adjust, keys, text_threshold)
    logger.info(f"Threshold Level: {threshold_level}")

    apply_morph = any(k in keys for k in do_morph)
    denoise = not any(k in keys for k in skip_denoise)

    fuzzy_pct = default_fuzzy_pct
    pixel_count_columns = default_pixel_count_columns
    pixel_count_rows = default_pixel_count_rows
    if k := dict_multi_get(edge_finding_adjust, keys):
        fuzzy_pct = k[0]
        pixel_count_columns = k[1]
        pixel_count_rows = k[2]
        logger.info(
            f"Adjusting Edge Finding: ({fuzzy_pct}, {pixel_count_columns}, {pixel_count_rows})"
        )

    deskew_before_crop_value = dict_multi_get(deskew_before_crop, keys)
    deskew_after_crop_value = dict_multi_get(deskew_after_crop, keys)

    skip_auto_deskew_list = [*skip_auto_deskew, *deskew_before_crop, *deskew_after_crop]
    execute_auto_deskew = not (
        prefix in skip_auto_deskew_list or idx in skip_auto_deskew_list
    )

    initial_crop_value = dict_multi_get(d=initial_crop, keys=keys, default=initial_crop_all)

    single_dimension_rescale_page = (
        idx in single_dimension_rescale or prefix in single_dimension_rescale
    )
    rotated_page = idx in rotated_standard_pages or prefix in rotated_standard_pages

    force_align = "Default"
    if any(k in keys for k in align_top_pages):
        force_align = "Top"
    elif any(k in keys for k in align_center_pages):
        force_align = "Center"
    elif any(k in keys for k in align_bottom_pages):
        force_align = "Bottom"

    if any(k in keys for k in blank_pages):
        logger.info(
            f"Creating Blank Proofing Image for idx {idx} prefix {prefix}"
        )
        processfiles.create_blank_proof(
            proofing_image_file=proofing_image,
            pre_ocr_target_file=pre_ocr_target_file,
            h_w_ratio=page_h_w_ratio,
        )
    else:
        logger.info(f"Creating Proofing Image for idx {idx} prefix {prefix}")

        processfiles.process_proofing_image(
            src=proofing_source_file_path,
            proofing_image=proofing_image,
            pre_ocr_target_file=pre_ocr_target_file,
            ocr_reader=None,
            threshold_level=threshold_level,
            skip_ocr=True,
            force_align=force_align,
            debug=debug,
            debug_path=debug_png_path,
            white_space_additional=white_space_additional_value,
            apply_morph=apply_morph,
            denoise=denoise,
            execute_auto_deskew=execute_auto_deskew,
            fuzzy_pct=fuzzy_pct,
            pixel_count_columns=pixel_count_columns,
            pixel_count_rows=pixel_count_rows,
            h_w_ratio=page_h_w_ratio,
            initial_crop_value=initial_crop_value,
            deskew_before_crop_value=deskew_before_crop_value,
            deskew_after_crop_value=deskew_after_crop_value,
            single_dimension_rescale_page=single_dimension_rescale_page,
            rotated_page=rotated_page,
        )

    if optimize_png:
        processfiles.run_optipng(proofing_image)


def process_proofing_image(
    idx,
    debug=False,
):
    logfile = pathlib.Path(logpath, f"process_proofing_image_{idx}.log")

    new_handler = logging.FileHandler(filename=logfile, mode="w", encoding="utf-8")
    new_handler.setFormatter(formatter)
    new_handler.setLevel(logging.DEBUG)
    new_handler.addFilter(ThreadFilter(get_ident()))
    processfiles.logger.addHandler(new_handler)

    parent = processing_dict[idx]
    for output in processing_dict[idx]["outputs"]:
        source_file_path = output.get(
            "override_source_file",
            parent.get(
                "grayscale_override_png_file",
                parent.get(
                    "grayscale_override_jpg_file",
                    parent.get(
                        "original_override_png_file",
                        parent.get(
                            "original_override_jpg_file",
                            parent["original_file"],
                        ),
                    ),
                ),
            ),
        )

        process_proofing_image_from_source(
            prefix=output["full_prefix"],
            idx=idx,
            proofing_source_file_path=source_file_path,
            proofing_image=output["proofing_image"],
            pre_ocr_target_file=output["pre_ocr_target_file"],
            debug=debug,
            optimize_png=True,
        )

    processfiles.logger.removeHandler(new_handler)

    return f"Completed: {idx}"

#### 5b. Execute Process


In [19]:
if 5 in skip_steps:
    logger.info("Skipping Step 5")
else:
    with ThreadPoolExecutor(max_workers=25) as executor:
        proofs = [
            executor.submit(process_proofing_image, idx, False)
            for idx in sorted(processing_dict.keys()) if not processing_dict[idx]["ignore"]
            if idx in [311]
        ]
        for future in as_completed(proofs):
            logger.info(future.result())

2024-12-20 07:09:48,324-root-INFO-1637285206.py-2-<module>-Skipping Step 5


In [20]:
processing_dict[329]

{'name': 'fourmenfarrago00belluoft_0330.jp2',
 'stem': 'fourmenfarrago00belluoft_0330',
 'original_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/fourmenfarrago00belluoft_jp2/fourmenfarrago00belluoft_0330.jp2'),
 'outputs': [{'full_prefix': 'p324',
   'full_page': True,
   'pre_ocr_target_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/pre_ocr_images_png/fourmenfarrago00belluoft_0330_p324.png'),
   'ocr_target_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_images_png/fourmenfarrago00belluoft_0330_p324.png'),
   'ocr_target_text_file': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0330_p324.txt'),
   'proofing_image': PosixPath('/home/linuxuser/PGDP/belloc-the-four-men/processing/proofing_images_png/fourmenfarrago00belluoft_0330_p324.png')}],
 'prefix': 'p324',
 'ignore': False,
 'blank': False,
 'skip_ocr_crop': False,
 'original_override_jpg_file': PosixPath('/home/linuxuser

### Step 6: N/A


### Step 7: Inspect proofing images for marginalia issues

In [21]:
if 7 in skip_steps:
    logger.info("Skipping Step 7")
else:
    proofing_images = list(sorted([f for f in proofing_images_png_path.iterdir() if f.is_file()]))
    #print("Proofing images: {}".format(proofing_images))

    for f in processing_dict:
        for d in processing_dict[f]["outputs"]:
            if (f not in blank_pages):
                processfiles.inspect_margins(d["proofing_image"], 1.5, 100, 100)


2024-12-20 07:09:48,347-root-INFO-3077651960.py-2-<module>-Skipping Step 7


### Step 8: Crop proofing images for OCR

#### 8a. Define Function

In [22]:
def crop_proofing_image(source_file: pathlib.Path, target_file: pathlib.Path, skip_ocr_crop: bool, debug=False):
    text = f"Source File: {source_file}"

    logfile = pathlib.Path(logpath, f"crop_ocr_image_{target_file.stem}.log")
    new_handler = logging.FileHandler(filename=logfile, mode="w", encoding="utf-8")
    new_handler.setFormatter(formatter)
    new_handler.setLevel(logging.DEBUG)
    new_handler.addFilter(ThreadFilter(get_ident()))
    processfiles.logger.addHandler(new_handler)
    logger.addHandler(new_handler)

    result = ""
    if skip_ocr_crop:
        shutil.copyfile(
            source_file.absolute().as_posix(),
            target_file.absolute().as_posix(),
        )
        result = f"Skipped: {text}"
    else:
        processfiles.crop_and_save(
            source_file_path=source_file,
            target_file_path=target_file,
            top=ocr_crop_top,
            bottom=ocr_crop_bottom,
            left=ocr_crop_left,
            right=ocr_crop_right,
            debug=debug,
            debug_path=debug_png_path,
        )
        result = f"Completed: {text}"

    processfiles.logger.removeHandler(new_handler)
    logger.removeHandler(new_handler)
    return result

#### 8b. Run over all source images

In [23]:
if 8 in skip_steps:
    print("Skipping!")
else:
    outputs = []

    for d in processing_dict:
        if not processing_dict[d]["ignore"]:
            t = processing_dict[d]["outputs"]
            for o in t:
                v = o
                o["idx"] = d
                if d in ocr_crop_skip:
                    o["skip_ocr_crop"] = True
                outputs.append(o)

    with ThreadPoolExecutor() as executor:
        crop = [
            executor.submit(
                crop_proofing_image,
                o["pre_ocr_target_file"],
                o["ocr_target_file"],
                o.get("skip_ocr_crop", False)
            )
            for o in outputs
            # if o["idx"] == 35
        ]
        for future in as_completed(crop):
            logger.info(future.result())

Skipping!


### Step 9: OCR the Text

#### 9a. Function

In [24]:
# TODO: Check back on EasyOCR, Keras-OCR, and Tesseract updates

def run_tesseract(source_image_file: pathlib.Path, target_text_file: pathlib.Path, blank, dpi=150):
    print(f"Processing: {source_image_file}")

    if blank:
        f = open(target_text_file, "w")
        f.write("[Blank Page]\n")
        f.close()
        return

    processfiles.run_tesseract(
        source_image_file=source_image_file,
        target_text_file=target_text_file,
        dpi=dpi,
    )
    return f"Completed: {target_text_file}"

#### 9b. Run over all images

In [25]:
if 9 in skip_steps:
    print("Skipping!")
else:
    outputs = []

    for d in processing_dict:
        if not processing_dict[d]["ignore"]:
            t = processing_dict[d]["outputs"]
            for o in t:
                v = o
                o["idx"] = d
                if d in blank_pages:
                    o["blank"] = True
                outputs.append(o)

    with ThreadPoolExecutor(max_workers=8) as executor:
        ocr_processes = [
            executor.submit(run_tesseract,
                o["ocr_target_file"],
                o["ocr_target_text_file"],
                o.get("blank", False),
                150
            )
            for o in outputs
            # if o["idx"] == 35
        ]
        for future in as_completed(ocr_processes):
            print(future.result())

Skipping!


### 10: Regex the text

#### 10a. Define regex functions

In [26]:
def run_regex(source_file_path, idx, find_replace_list):
    print(f"Processing: {source_file_path}")

    if idx in blank_pages:
        print(f"{source_file_path} is a blank page")
        return

    processfiles.replace_string_in_file(
        fpath=source_file_path,
        find_replace_list=find_replace_list,
        ignore_case=True
    )
    return f"Completed: {source_file_path}"

def run_regex_over_threads(find_replace_list):
    outputs = []

    for d in processing_dict:
        if not processing_dict[d]["ignore"]:
            t = processing_dict[d]["outputs"]
            for o in t:
                v = o
                o["idx"] = d
                if d in blank_pages:
                    o["blank"] = True
                outputs.append(o)

    with ThreadPoolExecutor(max_workers=50) as executor:
        thumbnails = [
            executor.submit(run_regex, o["ocr_target_text_file"], o["idx"], find_replace_list)
            for o in outputs
        ]
        for future in as_completed(thumbnails):
            print(future.result())

#### 10b. Capture Mismatched Dashes Prior to any changes to the OCR'd text

In [27]:
if 10 in skip_steps:
    print("Skipping!")
else:
    outputs = []
    for d in processing_dict:
        if not processing_dict[d]["ignore"]:
            t = processing_dict[d]["outputs"]
            for o in t:
                v = o
                o["idx"] = d
                if d in blank_pages:
                    o["blank"] = True
                outputs.append(o)

    ocr_list = list(sorted([o["ocr_target_text_file"] for o in outputs]))
    processfiles.find_mismatched_dashes(file_list=ocr_list)

Skipping!


##### Results here:

```
```

#### Quotation Marks

In [28]:
if 10 in skip_steps:
    print("Skipping!")
else:
    find_replace_list = [("“ ", '"'), ("“", '"'), (" ”", '"'), ("”", '"'), ("‘|’", "'")]
    run_regex_over_threads(find_replace_list)

Skipping!


#### Punctuation with space before it

In [29]:
if 10 in skip_steps:
    print("Skipping!")
else:
    find_replace_list = [(r" ([:;!\?])", "\\1")]
    run_regex_over_threads(find_replace_list)

Skipping!


#### Em-Dashes  

In [30]:
if 10 in skip_steps:
    print("Skipping!")
else:
    find_replace_list = [(" *— *", "--")]
    run_regex_over_threads(find_replace_list)




Skipping!


#### Em-Dash Cleanup

In [31]:
if 10 in skip_steps:
    print("Skipping!")
else:
    find_replace_list = [(" *--{1,2} *", "--"), ("(?<!\n)--\n *([^\\s]*) ", "--\\1\n")]
    run_regex_over_threads(find_replace_list)

Skipping!


In [32]:
if 10 in skip_steps:
    print("Skipping!")
else:
    find_replace_list = [("(?<!\n)\n(--[^\\s]*) ", "\\1\n"),]
    run_regex_over_threads(find_replace_list)

Skipping!


#### Common Hyphenated Words

In [33]:
punctuation = "[,;:!\\?\\.\"'\\-\\(\\)]"
common_endings = "s|d|r|er|ed|ing|ly"

def replace_list_for_common_ending(endings_list: str):
    endings_pipe = "|".join(endings_list)
    print (endings_pipe)
    return [
        (f"-\n({endings_pipe})[\n ]", "\\1\n"),
        (f"-\n({endings_pipe})({punctuation}{{1,4}})[\n ]", "\\1\\2\n"), # With Punctuation Following
        (f"-\n({endings_pipe})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\n"), # With Punctuation and then Nonwhitespace after punctuation
        (f"-\n({endings_pipe})({common_endings})[\n ]", "\\1\\2\n"), # common word endings
        (f"-\n({endings_pipe})({common_endings})({punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n"), # common word endins With Punctuation Following
        (f"-\n({endings_pipe})({common_endings})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n"), # common word endins With Punctuation Following and then Nonwhitespace after punctuation
    ]

def replace_list_for_common_beginning(beginnings_list: list[str]):
    beginnings_pipe = "|".join(beginnings_list)
    print (beginnings_pipe)
    return [
        (f"(?<=[^\\w])({beginnings_pipe})-\n([^\\s]+)[\n ]", "\\1\\2\n"),
    ]

def replace_list_for_beginning_alwaysjoin_list(beginnings_alwaysjoin_list: list[str]):
    regexlist = []
    for l in beginnings_alwaysjoin_list:
        prefix = l[0]
        suffix_pipe = "|".join(l[1])
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})[\n ]", "\\1\\2\n"))
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})({punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n")) # With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n")) # With Punctuation and then Nonwhitespace after punctuation
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})({common_endings})[\n ]", "\\1\\2\\3\n")) # common word endings
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})({common_endings})({punctuation}{{1,4}})[\n ]", "\\1\\2\\3\\4\n")) # common word endings With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix})-\n({suffix_pipe})({common_endings})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\\4\n")) # common word endings With Punctuation and then Nonwhitespace after punctuation

    print (regexlist)
    return regexlist

def replace_list_for_ending_alwaysjoin_list(endings_alwaysjoin_list: list[str]):
    regexlist = []
    for l in endings_alwaysjoin_list:
        suffix = l[0]
        prefix_pipe = "|".join(l[1])
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})[\n ]", "\\1\\2\n"))
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})({punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n")) # With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n")) # With Punctuation and then Nonwhitespace after punctuation
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})({common_endings})[\n ]", "\\1\\2\\3\n")) # common word endings
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})({common_endings}{punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n")) # common word endings With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix_pipe})-\n({suffix})({common_endings}{punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n")) # common word endings With Punctuation and then Nonwhitespace after punctuation

    print (regexlist)
    return regexlist



def replace_list_for_beginning_alwayshypenate_list(beginnings_alwayshypenate_list: list[str]):
    regexlist = []
    for l in beginnings_alwayshypenate_list:
        prefix = l[0]
        suffix_pipe = "|".join(l[1])
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})[\n ]", "\\1\\2\n"))
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})({punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n")) # With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})({punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n")) # With Punctuation and then Nonwhitespace after punctuation
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})({common_endings})[\n ]", "\\1\\2\\3\n")) # common word endings
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})({common_endings}{punctuation}{{1,4}})[\n ]", "\\1\\2\\3\n")) # common word endings With Punctuation Following
        regexlist.append((f"(?<=[^\\w])({prefix}-)\n({suffix_pipe})({common_endings}{punctuation}{{1,4}}[^\\s]+)[\n ]", "\\1\\2\\3\n")) # common word endings With Punctuation and then Nonwhitespace after punctuation

    print (regexlist)
    return regexlist


#### Regex to find additional partial words for auto-join to add to the list

##### Beginning
[^\s]*(?<!pro|pre|of|here|with|is|in|be|him|the|arch|book|one|no|sub|fur|every|human|trans|who|them|super|or|eco|how|lit|after|tri|inter|enter|an|six|over|to)-\n[^\s]+

##### Endings
(?<=-\n)[^\s]+

#### TODO: Common words to always join
```
```

#### TODO: process to  search text for hypenated version of words, if found, join together

In [34]:
import json
import pathlib

with open(pathlib.Path(bookpath, "../common_processing_scripts", "hyphenated-line-join.json").absolute().as_posix()) as f:
    json_data = json.load(f)
    json_data["beginnings"] = sorted(set(json_data["beginnings"]))
    json_data["endings"] = sorted(set(json_data["endings"]))
    hyphenated_line_beginning_join = json_data["beginnings"]
    hyphenated_line_ending_join = json_data["endings"]
    hyphenated_line_beginning_alwaysjoin = json_data["beginnings-alwaysjoin"]
    hyphenated_line_ending_alwaysjoin = json_data["endings-alwaysjoin"]
    hyphenated_line_beginnings_alwayshypenate = json_data["beginnings-alwayshypenate"]

In [35]:
# if 10 in skip_steps:
#     print("Skipping!")
# else:
#     find_replace_list = replace_list_for_common_ending(sorted(hyphenated_line_ending_join))
#     run_regex_over_threads(find_replace_list)

#     find_replace_list = replace_list_for_common_beginning(sorted(hyphenated_line_beginning_join))
#     run_regex_over_threads(find_replace_list)

#     find_replace_list = replace_list_for_beginning_alwaysjoin_list(sorted(hyphenated_line_beginning_alwaysjoin))
#     run_regex_over_threads(find_replace_list)

#     find_replace_list = replace_list_for_ending_alwaysjoin_list(sorted(hyphenated_line_ending_alwaysjoin))
#     run_regex_over_threads(find_replace_list)

#     find_replace_list = replace_list_for_beginning_alwayshypenate_list(sorted(hyphenated_line_beginnings_alwayshypenate))
#     run_regex_over_threads(find_replace_list)


#### Scanno words

{} > is

In [36]:
scanno_words = [
    (r"(?<!\w)IJ(?!\w)", "I"),
    ("1n", "in"),
    ("1s", "is"),
    ("1t", "it"),
    (r"I\]", "I"),
    ("([Tt])oa", "\\1o a"),
    ("([Aa])ta", "\\1t a"),
    (r"(?<!\w)J(?!\w)", "I"),
    (r"(?<!\w)\[(?!\w)", "I"),
    (r"(?<!\w)\|(?!\w)", "I"),
    (r"(?<!\w)1(?!\w)", "I"),
    (r"\[am", "I am"),
]

def process_scanno(srctgt):
    find_replace_list = [
        (rf"(?<=\s){srctgt[0]}(?=\s+)", srctgt[1]),
        (f"(?<=\\s){srctgt[0]}(?=\n)", srctgt[1]),
        (rf"(?<=\s+){srctgt[0]}(?=({punctuation}{{1,5}}))", srctgt[1]),
        (rf"(?<={punctuation}){srctgt[0]}(?=\s+)", srctgt[1]),
        (rf"(?<={punctuation}{{2}}){srctgt[0]}(?=\s+)", srctgt[1]),
        (rf"(?<={punctuation}{{3}}){srctgt[0]}(?=\s+)", srctgt[1]),
        (rf"(?<={punctuation}{{4}}){srctgt[0]}(?=\s+)", srctgt[1]),
        (
            f"(?<={punctuation}){srctgt[0]}(?=({punctuation}{{1,5}}))",
            srctgt[1],
        ),
        (
            f"(?<={punctuation}{{2}}){srctgt[0]}(?=({punctuation}{{1,5}}))",
            srctgt[1],
        ),
        (
            f"(?<={punctuation}{{3}}){srctgt[0]}(?=({punctuation}{{1,5}}))",
            srctgt[1],
        ),
        (
            f"(?<={punctuation}{{4}}){srctgt[0]}(?=({punctuation}{{1,5}}))",
            srctgt[1],
        ),
    ]
    run_regex_over_threads(find_replace_list)

In [37]:
if 10 in skip_steps:
    print("Skipping!")
else:
    for scanno in scanno_words:
        process_scanno(scanno)

Skipping!


In [38]:
# find mismatched dashes

# join hyphens across lines then run this

ocr_list = list(sorted([f for f in ocr_texts_path.iterdir() if f.is_file()]))
processfiles.find_mismatched_dashes(file_list=ocr_list)

[('arding-ly', 1, 'ardingly', 3),
 ('bos-ham', 1, 'bosham', 3),
 ('hors-ham', 1, 'horsham', 2)]

#### Checking Changes

`git diff --minimal -U0 --word-diff -U0 | grep "\[\|\]" > wordreplace.txt`

"[^\w]' \w"

check for numbers and letters mixed but not 1st 2nd 3rd etc

fix | > I

1$ > is

1t

symbols

### Step 11: Do Manual Regexes

```
# Manual Regexes

# Find hyphens except at end of file (\w+-)\n(?=\s*\w+)

# Non-Word Characters
# [^\w|\.|;|:|'|"| |\-|,|\(|\)|\?|!|&]|_

# Non-Ascii Characters
# [^\x00-\x7F]

# Mixed Case
# ([a-z]\w*[A-Z]\w*)|(?![A-Z]*?([^\w]|\n))([A-Z]\w*[A-Z]\w*)

# 1 as I
# \]|\[|[^\w]1(?!\d)

# Non-lowercase at start of page, ensure extra CR when requried
# (?<![\r\n])^[^a-z\n]

# lines that don't join up to prev paragraph properly
# (?<![A-Z]\.)\n\n([a-z].*?)\n$

# find brackets, replace with I
# find single J letters, replace with I

# Look for Il > ll

# ^.\n

\s(?!A|I|a)\w\s


[^\x00-\x7F|\.|;|:|'|"| |\-|,|\(|\)|\?|!|&|ń|ó|ś|é|ł|ż|ź|ć|ę|ö|ä|ą|Ǹ|ô|ā|ç|ō|=|\$|\*]|_

[^\x00-\x7F|\.|;|:|'|"| |\-|,|\(|\)|\?|!|&|ń|ó|ś|é|ł|ż|ź|ć|ę|ö|ä|ą|Ǹ|ô|ā|ç|ō|=|\$]|_
```

### Step 12: Copy Text and PNG to For Zip Folder

In [39]:
step_id = 12

for d in list(processing_dict.keys()):
    if not processing_dict[d]["ignore"]:
        for o in processing_dict[d]["outputs"]:
            ocr_target_text_file = o["ocr_target_text_file"]
            for_zip_text = pathlib.PosixPath(for_zip_path, f"{o['full_prefix']}.txt")
            o["for_zip_text"] = for_zip_text

            if step_id not in skip_steps:
                print(f"Copying {ocr_target_text_file} to {for_zip_text}")
                shutil.copyfile(src=ocr_target_text_file.as_posix(), dst=for_zip_text.as_posix())

            for_zip_png = pathlib.PosixPath(for_zip_path, f"{o['full_prefix']}.png")
            o["for_zip_png"] = for_zip_png

            if step_id not in skip_steps:
                shutil.copyfile(src=o["proofing_image"].as_posix(), dst=for_zip_png.as_posix())



Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0005_f001.txt to /home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/f001.txt
Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0006_f002.txt to /home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/f002.txt
Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0007_f003b.txt to /home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/f003b.txt
Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0008_f003p.txt to /home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/f003p.txt
Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_0009_f003.txt to /home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/f003.txt
Copying /home/linuxuser/PGDP/belloc-the-four-men/processing/ocr_text/fourmenfarrago00belluoft_00

In [ ]:
processing_dict

### Step 13: Copy Hi-Res to For Zip Folder

In [ ]:
for k in processing_dict:
    processing_dict[k]["for_zip_hi_res"] = set([])

In [ ]:
step_id = 13
flist = sorted(pathlib.PosixPath(hi_res_jpg_path).glob("*"))
for src_path in flist:
    if src_path.suffix in [".jpg", ".png"] and src_path.stem[0:1] == "i":
        if src_path.stem[0:2] == "i_":
            dst_path = pathlib.PosixPath(for_zip_path, src_path.name)
            if step_id not in skip_steps:
                logger.info(f"Copying {src_path.name} to {dst_path.name}")
                shutil.copyfile(src=src_path.as_posix(), dst=dst_path.as_posix())
        else:
            illustration_idx = str(src_path.stem[1:3])
            for k in processing_dict:
                if processing_dict[k]["stem"] in src_path.stem:
                    dst_path = pathlib.PosixPath(
                        for_zip_path,
                        "i_"
                        + processing_dict[k]["prefix"]
                        + "_"
                        + illustration_idx
                        + src_path.suffix,
                    )
                    if step_id not in skip_steps:
                        logger.info(f"Copying {src_path.name} to {dst_path.name}")
                        shutil.copyfile(
                            src=src_path.as_posix(), dst=dst_path.as_posix()
                        )
                    if "for_zip_hi_res" in processing_dict[k]:
                        l = list(processing_dict[k]["for_zip_hi_res"])
                        processing_dict[k]["for_zip_hi_res"] = set([*l, dst_path])
                    else:
                        processing_dict[k]["for_zip_hi_res"] = set([dst_path])

In [ ]:
processing_dict

In [42]:
img = processfiles.load_image(pathlib.Path("/home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/a001.png"))
img = processfiles.grayscale(img)
img = processfiles.thresh(img, 127)
processfiles.write_png(
    img,
    pathlib.Path("/home/linuxuser/PGDP/belloc-the-four-men/processing/for_zip/a001-new.png"),
)

libpng warning: iCCP: profile 'ICC profile': 'RGB ': RGB color space not permitted on grayscale PNG


### Step 14: Create Side-by-Side HTML to view everything together

In [43]:
step_id = 14
if step_id in skip_steps:
    logger.info(f"Skipping Step {step_id}")
else:
    processfiles.create_side_by_side_html(
        processing_dict=processing_dict,
        target_path=workpath,
    )

### Step 13: Copy Proofing Images and Text into single folder

sudo apt install p7zip-full p7zip-rar

```
rm for_upload_png_ocr.zip || echo "no for_upload_png_ocr zip"
rm for_upload_hi_res.zip || echo "no for_upload_hi_res zip"

rm -rf for_upload_png_ocr
mkdir for_upload_png_ocr
cp proofing_images_png/* for_upload_png_ocr
cp ocr_text/* for_upload_png_ocr
rm -rf for_upload_hi_res
mkdir for_upload_hi_res
cp hi_res_jpg/* for_upload_hi_res

cd for_upload_png_ocr
7z a -tzip ../for_upload_png_ocr.zip *
cd ../for_upload_hi_res
7z a -tzip ../for_upload_hi_res.zip *
cd ..

```